<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will use two observed February 2026 Search Console signals: search impressions as a volume signal, and click-through rate compared with average position as a CTR-fix signal.

The volume signal is linked to the quick-win idea: content with meaningful impressions may have an opportunity to improve. The CTR-versus-position signal is linked to the CTR-fix idea: content ranking well but receiving few clicks deserves review.

The rule gives every client-content pair one score, one reason code, and one action label. It uses only February data available at the decision moment. It does not use March data, future windows, product flags, or outcome labels.

In [ ]:
!pip -q install duckdb huggingface_hub pandas

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN missing. Colab Secrets mein access ON karo.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

feb_matches = files[
    files["file"].astype(str).str.contains("2026-02", regex=False)
]

FEB = feb_matches.iloc[0]["file"]

print("Setup complete")
print("FEB:", FEB)

Setup complete
FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet


In [ ]:
# Build one February row per client-content pair
base = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COALESCE(
        SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE),
        0
    ) AS gsc_impressions,

    COALESCE(
        SUM(gsc_clicks)
        FILTER (WHERE gsc_data_available IS TRUE),
        0
    ) AS gsc_clicks,

    AVG(gsc_avg_position)
        FILTER (
            WHERE gsc_data_available IS TRUE
            AND gsc_avg_position IS NOT NULL
        ) AS gsc_avg_position

FROM read_parquet('{FEB}')
WHERE month = '2026-02'
GROUP BY client_hash_id, content_hash_id
""").df()

base["ctr"] = np.where(
    base["gsc_impressions"] > 0,
    base["gsc_clicks"] / base["gsc_impressions"],
    np.nan
)

print("Rows in February base:", len(base))
display(base.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in February base: 321546


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,0.002062


In [ ]:
volume_buckets = con.sql(f"""
WITH volume AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COALESCE(
            SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE),
            0
        ) AS impressions
    FROM read_parquet('{FEB}')
    WHERE month = '2026-02'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN impressions = 0 THEN 'NO_IMPRESSIONS'
        WHEN impressions < 100 THEN 'LOW_1_TO_99'
        WHEN impressions < 1000 THEN 'MEDIUM_100_TO_999'
        ELSE 'HIGH_1000_PLUS'
    END AS volume_bucket,
    COUNT(*) AS n
FROM volume
GROUP BY 1
ORDER BY
    CASE volume_bucket
        WHEN 'NO_IMPRESSIONS' THEN 1
        WHEN 'LOW_1_TO_99' THEN 2
        WHEN 'MEDIUM_100_TO_999' THEN 3
        ELSE 4
    END
""").df()

print("Volume signal n =", int(volume_buckets["n"].sum()))
display(volume_buckets)

Volume signal n = 321546


,volume_bucket,n
0,NO_IMPRESSIONS,167987
1,LOW_1_TO_99,73237
2,MEDIUM_100_TO_999,47015
3,HIGH_1000_PLUS,33307


**Volume verdict: CONFIRMED.** The bucket table shows that impressions are observed and vary across content rows. This is a usable signal for a volume-based quick-win baseline, although the threshold is directional rather than causal.

In [ ]:
ctr_position_buckets = con.sql(f"""
WITH metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COALESCE(
            SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE),
            0
        ) AS impressions,

        COALESCE(
            SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE),
            0
        ) AS clicks,

        AVG(gsc_avg_position)
            FILTER (
                WHERE gsc_data_available IS TRUE
                AND gsc_avg_position IS NOT NULL
            ) AS avg_position

    FROM read_parquet('{FEB}')
    WHERE month = '2026-02'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    CASE
        WHEN impressions = 0 OR avg_position IS NULL
            THEN 'NO_USABLE_GSC'
        WHEN avg_position <= 10
             AND clicks::DOUBLE / NULLIF(impressions, 0) < 0.02
            THEN 'TOP10_LOW_CTR'
        WHEN avg_position <= 10
             AND clicks::DOUBLE / NULLIF(impressions, 0) >= 0.02
            THEN 'TOP10_OK_CTR'
        WHEN avg_position <= 20
            THEN 'POSITIONS_11_TO_20'
        ELSE 'POSITION_21_PLUS'
    END AS ctr_position_bucket,
    COUNT(*) AS n
FROM metrics
GROUP BY 1
ORDER BY n DESC
""").df()

print("CTR-position signal n =", int(ctr_position_buckets["n"].sum()))
display(ctr_position_buckets)


CTR-position signal n = 321546


,ctr_position_bucket,n
0,NO_USABLE_GSC,167987
1,TOP10_LOW_CTR,92088
2,POSITIONS_11_TO_20,31700
3,POSITION_21_PLUS,26718
4,TOP10_OK_CTR,3053


**CTR-versus-position verdict: MIXED.** The signal exists, but many rows lack usable Search Console data, so it should not be treated as reliable for every row.

## 2. Build the ranked queue (writes the CSV)

Rule:

- `CTR_FIX`: average position is 10 or better and CTR is below 2%.
- `QUICK_WIN_VOLUME`: impressions are at least 1,000 and average position is between 11 and 20.
- `MONITOR`: rows that do not meet either priority condition.

The score gives CTR-fix rows the highest priority, then volume-based quick wins. The score is only a directional decision-support score, not a prediction of future performance.

In [ ]:
queue = base.copy()

queue["reason_code"] = np.select(
    [
        (
            (queue["gsc_avg_position"] <= 10)
            & (queue["ctr"] < 0.02)
            & (queue["gsc_impressions"] > 0)
        ),
        (
            (queue["gsc_impressions"] >= 1000)
            & (queue["gsc_avg_position"] > 10)
            & (queue["gsc_avg_position"] <= 20)
        )
    ],
    [
        "CTR_FIX",
        "QUICK_WIN_VOLUME"
    ],
    default="MONITOR"
)

queue["action_label"] = np.select(
    [
        queue["reason_code"] == "CTR_FIX",
        queue["reason_code"] == "QUICK_WIN_VOLUME"
    ],
    [
        "REVIEW_CTR",
        "REVIEW_CONTENT"
    ],
    default="MONITOR"
)

queue["score"] = np.select(
    [
        queue["reason_code"] == "CTR_FIX",
        queue["reason_code"] == "QUICK_WIN_VOLUME"
    ],
    [
        100 + np.minimum(30, np.log1p(queue["gsc_impressions"])),
        60 + np.minimum(30, np.log1p(queue["gsc_impressions"]))
    ],
    default=np.minimum(20, np.log1p(queue["gsc_impressions"]))
)

queue = queue.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action_label",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position"
]

queue = queue[output_columns]

import os
os.makedirs("work/outputs", exist_ok=True)

csv_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(csv_path, index=False)

print("Queue rows:", len(queue))
print("CSV written to:", csv_path)
display(queue.head(10))


Queue rows: 321546
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action_label,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,112.222940,CTR_FIX,REVIEW_CTR,203401.0,2.0,0.000010,4.967059
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,112.184078,CTR_FIX,REVIEW_CTR,195648.0,1.0,0.000005,2.488437
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,112.175381,CTR_FIX,REVIEW_CTR,193954.0,0.0,0.000000,3.844678
3,4,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,112.027568,CTR_FIX,REVIEW_CTR,167303.0,3310.0,0.019784,2.923168
4,5,client_73cda7b4e4f265ea,content_e241d6415ac9e534,112.008554,CTR_FIX,REVIEW_CTR,164152.0,401.0,0.002443,2.925926
5,6,client_62f4a7e64f5e0096,content_b99ea6861864dea5,111.987295,CTR_FIX,REVIEW_CTR,160699.0,273.0,0.001699,3.715542
6,7,client_62f4a7e64f5e0096,content_f107e54b10b43725,111.958662,CTR_FIX,REVIEW_CTR,156163.0,883.0,0.005654,3.095657
7,8,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,111.947969,CTR_FIX,REVIEW_CTR,154502.0,2508.0,0.016233,4.240872
8,9,client_62f4a7e64f5e0096,content_acbcc847f8996314,111.906703,CTR_FIX,REVIEW_CTR,148256.0,239.0,0.001612,3.907896
9,10,client_e547b89c05043229,content_c9a0c2fdbdbfb562,111.865102,CTR_FIX,REVIEW_CTR,142215.0,1605.0,0.011286,1.882000


## 3. Top-20 review

The top twenty rows are reviewed as individual decision-support suggestions. Each row has an action, one reason code, a confidence note, and a condition that could make the recommendation wrong.

In [ ]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "CTR_FIX":
        if row["gsc_impressions"] >= 1000:
            return "stronger signal: usable volume and top-10 low CTR"
        return "lower confidence: top-10 low CTR but limited volume"
    if row["reason_code"] == "QUICK_WIN_VOLUME":
        return "moderate signal: high volume with position 11-20"
    return "weak signal: no priority rule matched"

def wrong_if(row):
    if row["reason_code"] == "CTR_FIX":
        return "wrong if tracking is incomplete, impressions are too small, or the average position is not representative"
    if row["reason_code"] == "QUICK_WIN_VOLUME":
        return "wrong if high impressions do not represent meaningful opportunity or the content cannot be changed"
    return "wrong if the available GSC data is stale or incomplete"

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

for _, row in top20.iterrows():
    print(
        f'{int(row["rank"])}. '
        f'action={row["action_label"]}; '
        f'reason={row["reason_code"]}; '
        f'confidence={row["confidence_note"]}; '
        f'wrong if={row["what_would_make_it_wrong"]}'
    )


1. action=REVIEW_CTR; reason=CTR_FIX; confidence=stronger signal: usable volume and top-10 low CTR; wrong if=wrong if tracking is incomplete, impressions are too small, or the average position is not representative
2. action=REVIEW_CTR; reason=CTR_FIX; confidence=stronger signal: usable volume and top-10 low CTR; wrong if=wrong if tracking is incomplete, impressions are too small, or the average position is not representative
3. action=REVIEW_CTR; reason=CTR_FIX; confidence=stronger signal: usable volume and top-10 low CTR; wrong if=wrong if tracking is incomplete, impressions are too small, or the average position is not representative
4. action=REVIEW_CTR; reason=CTR_FIX; confidence=stronger signal: usable volume and top-10 low CTR; wrong if=wrong if tracking is incomplete, impressions are too small, or the average position is not representative
5. action=REVIEW_CTR; reason=CTR_FIX; confidence=stronger signal: usable volume and top-10 low CTR; wrong if=wrong if tracking is incomplete

## 4. Weak picks + leakage check

Some recommendations are weak when they depend on low volume or missing Search Console fields. I will inspect these rows separately instead of treating every ranked item as equally reliable.

The final queue uses only February Search Console fields. No March outcome, future window, product flag, client name, URL, or label-derived field is used.


In [ ]:
weak_picks = queue[
    (queue["gsc_impressions"] < 100)
    | (queue["gsc_avg_position"].isna())
    | (queue["gsc_impressions"] == 0)
].head(10).copy()

print("Weak-pick examples:", len(weak_picks))
display(weak_picks)

required_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action_label"
]

missing_columns = [
    col for col in required_columns
    if col not in queue.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# Actual forbidden inputs, not vague words in column names
forbidden_columns = [
    col for col in queue.columns
    if col.lower() in [
        "march_ga4_sessions",
        "label",
        "future_window",
        "product_flag"
    ]
]

print("Forbidden columns found:", forbidden_columns)
assert not forbidden_columns, f"Leakage columns found: {forbidden_columns}"

print("Leakage check: PASS")
print("Source window: February 2026 only")
print("Future label used: NO")
print("Product flags used: NO")

Weak-pick examples: 10


,rank,client_hash_id,content_hash_id,score,reason_code,action_label,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
51626,51627,client_62f4a7e64f5e0096,content_f3b4217db333cf59,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,2.163333
51627,51628,client_62f4a7e64f5e0096,content_4dedd5901951b4f7,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,4.347159
51628,51629,client_62f4a7e64f5e0096,content_c9314439e6c529bb,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,1.593963
51629,51630,client_62f4a7e64f5e0096,content_4c3585560e99086b,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,1.784821
51630,51631,client_62f4a7e64f5e0096,content_a33df40d37333468,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,3.983627
51631,51632,client_62f4a7e64f5e0096,content_61643fc2fa8954e0,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,9.611470
51632,51633,client_73cda7b4e4f265ea,content_df791d6625cb23c5,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,4.689881
51633,51634,client_73cda7b4e4f265ea,content_668e899601755288,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,2.801852
51634,51635,client_73cda7b4e4f265ea,content_ac432d8abef868fc,104.60517,CTR_FIX,REVIEW_CTR,99.0,1.0,0.010101,7.252160
51635,51636,client_73cda7b4e4f265ea,content_ac0cdd5dbf6c1977,104.60517,CTR_FIX,REVIEW_CTR,99.0,0.0,0.000000,5.756013


Forbidden columns found: []
Leakage check: PASS
Source window: February 2026 only
Future label used: NO
Product flags used: NO


## Self-check

- Two observed signals were checked with visible bucket tables and n values.
- At least one signal, volume, is linked to the quick-win flag idea.
- Every row receives one score, one reason code, and one action label.
- The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- The top twenty rows have action, reason, confidence, and wrong-if notes.
- Weak picks and limitations were reviewed.
- No March data, future window, product flag, or label-derived input was used.
- The notebook runs top to bottom without errors.
- The notebook is committed under `work/notebooks/w04_baseline_score.ipynb`.
